# 02 — Star Schema Construction & Load to PostgreSQL

**Purpose.** Transform the cleaned Olist flat tables into a **star schema**
in pandas, then load it into PostgreSQL using `src.db`.

**Why this notebook exists.** The raw tables are normalised and join-keyed —
fine for storage, but slow and error-prone for analytics. A star schema
denormalises the model into one **fact** table surrounded by **dimension**
tables, so every BI query is a single join-set away. This notebook is where
that transformation happens and where it lands in the warehouse.

**Flow.**
1. **Setup** — load all raw tables into pandas DataFrames.
2. **Build dimension tables** — `dim_customer`, `dim_product`, `dim_seller`,
   `dim_date`, `dim_geolocation`.
3. **Build the fact table** — `fact_order_item` at order-line grain, with
   payments / reviews / delivery dates denormalised onto each line.
4. **Load to PostgreSQL** — run the DDL, append each table, verify with
   `COUNT(*)`.
5. **Schema documentation** — an ER diagram and the rationale for the design.

*Inputs:* `data/raw/*.csv` (9 Olist tables), `sql/00_create_star_schema.sql`
*Outputs:* populated `dim_*` and `fact_order_item` tables in Postgres.

## 0. Setup

Imports, plus a one-shot load of every raw Olist CSV via `load_all_tables`
(which already parses the `*_date` / `*_timestamp` columns for `orders` and
`order_reviews`). We also grab the SQLAlchemy `engine` from `src.db` and the
project paths from `src.config`.

In [ ]:
from __future__ import annotations

import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sqlalchemy import text

# Make the project's `src` package importable when the notebook runs from notebooks/.
sys.path.append('..')

from src.config import PROJECT_ROOT, RAW_DATA_DIR  # noqa: E402
from src.data_loader import load_all_tables        # noqa: E402
from src.db import get_engine, load_df_to_sql, run_script  # noqa: E402

# Surface the ETL progress logs inline (info-level) so each load is visible.
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

SQL_DIR = PROJECT_ROOT / 'sql'

# One shared engine for the whole notebook (single connection pool).
engine = get_engine()

# Load every raw table once. `tables` is a {name: DataFrame} dict; convenience
# handles below keep the merges readable.
tables = load_all_tables(RAW_DATA_DIR)

orders               = tables['orders']
order_items          = tables['order_items']
order_payments       = tables['order_payments']
order_reviews        = tables['order_reviews']
products             = tables['products']
sellers              = tables['sellers']
customers            = tables['customers']
geolocation          = tables['geolocation']
category_translation = tables['product_category_translation']

print(f'Loaded {len(tables)} tables:')
for name, df in tables.items():
    print(f'  {name:30s} {len(df):>10,} rows')

## 1. Build Dimension Tables

Each dimension is built independently from its source table(s). The target
column names and types match `sql/00_create_star_schema.sql` so the frames
drop straight into the DDL with `if_exists='append'`.

### dim_customer

Direct 1:1 copy from `olist_customers_dataset`. Grain: one row per
`customer_id` (Olist's per-order identifier). `customer_unique_id` is the
*true* customer and lets us count distinct buyers / compute repeat-purchase
metrics downstream (notebook 03).

In [ ]:
dim_customer = customers.copy()

# Enforce the DDL column order so the append matches the table layout exactly.
dim_customer = dim_customer[[
    'customer_id',
    'customer_unique_id',
    'customer_zip_code_prefix',
    'customer_city',
    'customer_state',
]]

print(f'dim_customer: {len(dim_customer):,} rows x {dim_customer.shape[1]} cols')
assert dim_customer['customer_id'].is_unique, 'customer_id must be unique (PK)'
dim_customer.head()

### dim_product

Merge `products` with the English category translation so analysts never have
to look up the mapping. We drop the Portuguese `product_category_name` column
(the English one supersedes it) and derive `product_volume_cm3 = length ×
height × width`.

Two notes:
- The source CSV misspells `..._lenght`; we rename to the DDL spelling
  `..._length` so the columns match the table.
- Missing English categories (a few stragglers with no translation row) are
  filled with `'unknown'` rather than left null, so they don't silently drop
  out of category filters.

The DDL defines `product_volume_cm3` as a `GENERATED ALWAYS AS (...) STORED`
column, so Postgres will recompute it from L/H/W on insert — we still include
it here for the in-pandas analytical frame, but we exclude it at load time
(see §3).

In [ ]:
# Fix the source CSV's misspelled length columns to match the DDL.
dim_product = products.rename(columns={
    'product_name_lenght': 'product_name_length',
    'product_description_lenght': 'product_description_length',
})

# Left join keeps every product even if no translation row exists.
shape_before = dim_product.shape
dim_product = dim_product.merge(
    category_translation,
    on='product_category_name',
    how='left',
)
shape_after = dim_product.shape
print(f'merge w/ translation: {shape_before} -> {shape_after} '
      f'(rows unchanged = {shape_before[0] == shape_after[0]} left join)')
assert dim_product.shape[0] == shape_before[0], 'left join must not drop products'

# English category replaces Portuguese; fill the few untranslated stragglers.
dim_product['product_category_name_english'] = (
    dim_product['product_category_name_english'].fillna('unknown')
)

# Derived measure: physical volume in cm^3.
dim_product['product_volume_cm3'] = (
    dim_product['product_length_cm']
    * dim_product['product_height_cm']
    * dim_product['product_width_cm']
)

# Drop the Portuguese column now that English is in place.
dim_product = dim_product.drop(columns=['product_category_name'])

# DDL column order.
dim_product = dim_product[[
    'product_id',
    'product_category_name_english',
    'product_name_length',
    'product_description_length',
    'product_photos_qty',
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm',
    'product_volume_cm3',
]]

print(f'dim_product: {len(dim_product):,} rows x {dim_product.shape[1]} cols')
assert dim_product['product_id'].is_unique, 'product_id must be unique (PK)'
print(f"untranslated categories filled with 'unknown': "
      f"{(dim_product['product_category_name_english'] == 'unknown').sum():,}")
dim_product.head()

### dim_seller

Direct 1:1 copy from `olist_sellers_dataset`. Grain: one row per `seller_id`.
Mirrors `dim_customer` so customer-vs-seller geography is symmetric.

In [ ]:
dim_seller = sellers.copy()

dim_seller = dim_seller[[
    'seller_id',
    'seller_zip_code_prefix',
    'seller_city',
    'seller_state',
]]

print(f'dim_seller: {len(dim_seller):,} rows x {dim_seller.shape[1]} cols')
assert dim_seller['seller_id'].is_unique, 'seller_id must be unique (PK)'
dim_seller.head()

### dim_date

Generate a calendar from the **min to max** of `order_purchase_timestamp`
using `pd.date_range`, then explode each date into the day/month/quarter/
year/day-of-week/is_weekend/month_name columns the dashboards slice on. This
pre-exploded calendar means no `EXTRACT(...)` date functions are needed in any
query.

Grain: one row per calendar day; `date_id` = the calendar date (the fact
table's `date_id` is the order's purchase date).

In [ ]:
purchase_ts = orders['order_purchase_timestamp'].dropna()
start = purchase_ts.min().normalize()
end   = purchase_ts.max().normalize()
print(f'purchase timestamp window: {start:%Y-%m-%d}  ->  {end:%Y-%m-%d} '
      f'({(end - start).days + 1} days)')

# One row per day across the full purchase window (inclusive both ends).
dates = pd.date_range(start=start, end=end, freq='D')

dim_date = pd.DataFrame({'date_id': dates})
dim_date['day']          = dim_date['date_id'].dt.day
dim_date['month']        = dim_date['date_id'].dt.month
dim_date['quarter']      = dim_date['date_id'].dt.quarter
dim_date['year']         = dim_date['date_id'].dt.year
dim_date['day_of_week']  = dim_date['date_id'].dt.day_name()
dim_date['is_weekend']   = dim_date['date_id'].dt.dayofweek >= 5   # Sat=5, Sun=6
dim_date['month_name']   = dim_date['date_id'].dt.month_name()
# date_id is the DDL primary key (DATE); keep it as a date, not a timestamp.
dim_date['date_id'] = dim_date['date_id'].dt.date

print(f'dim_date: {len(dim_date):,} rows x {dim_date.shape[1]} cols')
assert dim_date['date_id'].is_unique, 'date_id must be unique (PK)'
dim_date.head()

### dim_geolocation

The raw `geolocation` table is ~1M rows but only ~19k distinct zip prefixes
— each prefix carries ~50 (lat, lng) points for different neighbourhoods.
Loading it raw would fan every zip-keyed join out ~50×. We **aggregate to one
representative row per zip prefix**: mean lat/lng (a centroid) plus the modal
city/state. The EDA notebook (§5.5) documents this ~50× reduction.

In [ ]:
def _mode(series: pd.Series):
    """Most frequent value (ties broken by first occurrence)."""
    m = series.mode(dropna=True)
    return m.iloc[0] if not m.empty else np.nan

raw_geo_rows = len(geolocation)
shape_before = geolocation.shape

dim_geolocation = (
    geolocation
    .groupby('geolocation_zip_code_prefix', as_index=False)
    .agg(
        geolocation_lat=('geolocation_lat', 'mean'),    # centroid latitude
        geolocation_lng=('geolocation_lng', 'mean'),    # centroid longitude
        geolocation_city=('geolocation_city', _mode),   # most common city
        geolocation_state=('geolocation_state', _mode), # most common state
    )
)
print(f'aggregate geolocation: {shape_before} -> {dim_geolocation.shape} '
      f'(collapse factor {raw_geo_rows / len(dim_geolocation):.0f}x)')
assert dim_geolocation['geolocation_zip_code_prefix'].is_unique, \
    'zip prefix must be unique after aggregation'

print(f'dim_geolocation: {len(dim_geolocation):,} rows x {dim_geolocation.shape[1]} cols')
dim_geolocation.head()

All five dimensions are built. Quick summary before we move to the fact:

In [ ]:
dims = {
    'dim_customer': dim_customer,
    'dim_product':  dim_product,
    'dim_seller':   dim_seller,
    'dim_date':     dim_date,
    'dim_geolocation': dim_geolocation,
}
for name, df in dims.items():
    print(f'  {name:18s} {len(df):>10,} rows')

## 2. Build Fact Table

Grain: **one row per order line** (`order_id`, `order_item_id`). We start from
`order_items` and *left-join* everything else onto it so no line item is ever
dropped — every order line carries the full context (customer, product,
seller, date, payment, review, delivery dates) needed for a single-pass
analytical query.

### 2a. Aggregate order_payments per order

**Why aggregation is needed:** `order_payments` is at a *finer* grain than
the order — one order can have multiple payment rows (sequential payments,
e.g. split across two vouchers, or a voucher + credit card). We confirmed
this during EDA: ~3k orders have more than one payment row. Joining payments
raw onto `order_items` would **fan the fact table out** (duplicate line items
per payment row) and double-count revenue.

We collapse each order's payments to a single row:
- `payment_value` — **sum** (total the customer paid across all payment rows).
- `payment_type` — the **most common** type for that order (the dominant
  instrument; a sensible single label per order).
- `payment_installments` — the **max** (represents the longest instalment
  plan in effect; conservative for affordability analysis).

In [ ]:
def _mode(series: pd.Series):
    """Most frequent value (ties broken by first occurrence)."""
    m = series.mode(dropna=True)
    return m.iloc[0] if not m.empty else np.nan

payments_agg = (
    order_payments
    .groupby('order_id', as_index=False)
    .agg(
        payment_value=('payment_value', 'sum'),
        payment_type=('payment_type', _mode),
        payment_installments=('payment_installments', 'max'),
    )
)
print(f'order_payments: {len(order_payments):,} rows  ->  '
      f'payments_agg: {len(payments_agg):,} rows (one per order)')
payments_agg.head()

### 2b. Aggregate order_reviews per order

**Why aggregation is needed:** an order can have more than one review (a
customer can update their rating over time). ~550 orders have multiple review
rows. Joining reviews raw would again fan out the fact table.

**Choice — mean vs first:** we take the **mean** `review_score`. A mean is a
better single-number summary of a customer's evolving satisfaction than an
arbitrary "first" pick: if a customer rated 2 then revised to 5 after a
service-recovery, the mean (3.5, rounded to 4) reflects both signals, whereas
`first` would freeze the dissatisfaction. We round to an int to match the
DDL's `INT` column. (If the order has only one review — the common case — the
mean equals that score exactly.)

In [ ]:
reviews_agg = (
    order_reviews
    .groupby('order_id', as_index=False)
    .agg(review_score=('review_score', 'mean'))
)
# Round to int to match the DDL (INT); preserves the original 1-5 scale.
reviews_agg['review_score'] = reviews_agg['review_score'].round().astype('Int64')

print(f'order_reviews: {len(order_reviews):,} rows  ->  '
      f'reviews_agg: {len(reviews_agg):,} rows (one per order)')
reviews_agg.head()

### 2c. Assemble fact_order_item

Chain of **left joins** onto `order_items` — left, because we want to keep
*every* line item even when the order has no payment / review / delivery
record. Each merge asserts its before/after shape and logs the row delta so a
fan-out (a silent many-to-many) can never slip through unnoticed.

In [ ]:
raw_item_count = len(order_items)
fact = order_items.copy()
print(f'start: fact_order_item grain = {raw_item_count:,} order lines')

def _log_merge(df, before, label):
    """Assert row count is unchanged (no fan-out / row loss) and log it."""
    d = len(df) - before
    assert len(df) == before, f'{label} changed row count: {before} -> {len(df)} (delta {d})'
    print(f'  {label:38s} {before:>8,} -> {len(df):>8,} rows  (left join, rows preserved)')

# (1) orders -> customer_id, order_status, purchase timestamp, delivery dates.
before = len(fact)
fact = fact.merge(
    orders[[
        'order_id', 'customer_id', 'order_status',
        'order_purchase_timestamp',
        'order_delivered_customer_date',
        'order_estimated_delivery_date',
    ]],
    on='order_id',
    how='left',
)
_log_merge(fact, before, 'merge orders')

# (2) aggregated payments per order (one row per order — no fan-out).
before = len(fact)
fact = fact.merge(payments_agg, on='order_id', how='left')
_log_merge(fact, before, 'merge payments_agg')

# (3) aggregated reviews per order (one row per order — no fan-out).
before = len(fact)
fact = fact.merge(reviews_agg, on='order_id', how='left')
_log_merge(fact, before, 'merge reviews_agg')

print(f'\nfinal fact shape: {fact.shape}')

Finalise the fact table: derive `date_id` from the purchase timestamp,
coerce the delivery/estimate columns to `DATE` (DDL wants `DATE`, not
timestamp), and select the columns in DDL order. Two columns are deliberately
excluded from the load:
- `order_item_pk` — a `BIGSERIAL` in Postgres, generated by the DB.
- `product_volume_cm3` lives in `dim_product`, not the fact; mentioned for
  reference (it is `GENERATED ALWAYS AS (...) STORED`).

Note: the source column `order_delivered_customer_date` is renamed to the DDL
name `delivered_customer_date` so the append maps cleanly to the table.

In [ ]:
# date_id = the order's purchase date (FK into dim_date).
fact['date_id'] = fact['order_purchase_timestamp'].dt.date

# DDL delivery/estimate columns are DATE; normalise timestamps to date.
for col in ['order_delivered_customer_date', 'order_estimated_delivery_date']:
    fact[col] = pd.to_datetime(fact[col], errors='coerce').dt.date

# Source names the delivered-customer column 'order_delivered_customer_date';
# the DDL column is 'delivered_customer_date' — rename to match the table.
fact = fact.rename(columns={
    'order_delivered_customer_date': 'delivered_customer_date',
})

# Select + order columns to match the DDL exactly (excluding order_item_pk,
# which is BIGSERIAL and auto-generated by Postgres).
fact_order_item = fact[[
    'order_id',
    'order_item_id',
    'customer_id',
    'product_id',
    'seller_id',
    'date_id',
    'price',
    'freight_value',
    'payment_value',
    'payment_type',
    'payment_installments',
    'review_score',
    'order_status',
    'delivered_customer_date',
    'order_estimated_delivery_date',
]].copy()

# payment_installments is INT in the DDL; pandas merge may have made it float.
fact_order_item['payment_installments'] = (
    fact_order_item['payment_installments'].astype('Int64')
)

fact_order_item.head()

### 2d. Validation

Sanity check: the fact table must have **exactly as many rows as the raw
`order_items`** — no more (no fan-out), no fewer (no dropped lines). We also
confirm the composite key `(order_id, order_item_id)` stays unique.

In [ ]:
n_fact = len(fact_order_item)
print(f'raw order_items : {raw_item_count:,} rows')
print(f'fact_order_item : {n_fact:,} rows')
assert n_fact == raw_item_count, \
    f'row-count mismatch! fact should equal raw order_items ({raw_item_count})'

composite_unique = (
    fact_order_item[['order_id', 'order_item_id']]
    .drop_duplicates().shape[0] == n_fact
)
assert composite_unique, '(order_id, order_item_id) must stay unique'
print('PASS: fact grain == raw order_items grain; composite key unique.')

## 3. Load to PostgreSQL

Now land the built frames in the warehouse. The order matters:

1. **Run the DDL** (`sql/00_create_star_schema.sql`) first — it drops and
   recreates the empty tables with the right columns, types, PKs, FKs, and
   indexes. It is wrapped in a transaction and is idempotent.
2. **Append** each dimension and the fact (`if_exists='append'`) so the
   `BIGSERIAL`/`GENERATED` columns and FKs are honoured by Postgres.
3. **Verify** with `COUNT(*)` on each table.

> **DB dependency.** Sections 3 onward require a running Postgres reachable
> via `.env` (see `.env.example`). If the connection check below fails, start
> Postgres and re-run from here — sections 0–2 are pure pandas and need no DB.

In [ ]:
from src.db import test_connection

assert test_connection(), \
    'Cannot reach Postgres — check .env (PG_HOST/PG_PORT/...) and that the service is running.'
print('Postgres connection OK.')

### 3a. Create the star schema (DDL)

`run_script` splits the file on `;` and runs every statement inside one
transaction — so the whole DDL is applied atomically.

In [ ]:
run_script(str(SQL_DIR / '00_create_star_schema.sql'))
print('star schema (re)created: 5 dimensions + fact_order_item')

### 3b. Load each table

Load dimensions first (FK targets must exist before the fact), then the fact.
Each call appends into the DDL-created table. Note `dim_product` is loaded
**without** `product_volume_cm3` — that column is `GENERATED ALWAYS AS
(length*height*width) STORED` in Postgres, so providing it would raise an
error; the DB computes it from the inputs.

In [ ]:
# Order matters: dimensions (FK targets) before the fact.
load_order = [
    ('dim_customer',     dim_customer),
    ('dim_product',      dim_product.drop(columns=['product_volume_cm3'])),  # DB-generated
    ('dim_seller',       dim_seller),
    ('dim_date',         dim_date),
    ('dim_geolocation',  dim_geolocation),
    ('fact_order_item',  fact_order_item),
]

results = {}
for table_name, df in load_order:
    rows = load_df_to_sql(df, table_name, if_exists='append')
    results[table_name] = rows
    print(f'  loaded {table_name:18s} -> {rows:>8,} rows')

print(f'\nloaded {len(results)} tables, '
      f'{sum(results.values()):,} rows total.')

### 3c. Verify with COUNT(*)

Round-trip check: query each table's row count and confirm it matches the
DataFrame we loaded. A mismatch means rows were dropped/added during load
(e.g. a PK/FK constraint silently rejected some).

In [ ]:
expected = {
    'dim_customer':    len(dim_customer),
    'dim_product':     len(dim_product),
    'dim_seller':      len(dim_seller),
    'dim_date':        len(dim_date),
    'dim_geolocation': len(dim_geolocation),
    'fact_order_item': len(fact_order_item),
}

verify_rows = []
with engine.connect() as conn:
    for table_name, n_expected in expected.items():
        n_db = conn.execute(text(f'SELECT COUNT(*) FROM {table_name}')).scalar()
        ok = (n_db == n_expected)
        verify_rows.append({
            'table': table_name,
            'expected': n_expected,
            'in_db': n_db,
            'match': 'PASS' if ok else 'FAIL',
        })
        if not ok:
            print(f'  MISMATCH {table_name}: expected {n_expected}, db has {n_db}')

verify_df = pd.DataFrame(verify_rows)
assert (verify_df['match'] == 'PASS').all(), 'one or more table counts did not match!'
verify_df

## 4. Schema Documentation

### ER diagram (fact ↔ dimensions)

```
                          dim_geolocation
                        (zip prefix, lat, lng,
                         city, state)
                              │  (zip prefix lookup;
                               │  not a hard FK)
        dim_customer           │             dim_seller
      ┌──────────────┐         │           ┌──────────────┐
      │ customer_id ◄┼────┐    │      ┌───►│ seller_id    │
      │ unique_id    │    │    │      │    │ zip/city/st  │
      │ zip/city/st  │    │    │      │    └──────────────┘
      └──────────────┘    │    │      │
                          │    │      │
                     ┌────┴────┴──────┴─────┐         dim_product
                     │   fact_order_item    │       ┌──────────────┐
                     │ order_item_pk (PK)   │       │ product_id   │
                     │ order_id             │  ┌───►│ category_en  │
                     │ order_item_id        │  │    │ weight/dims  │
                     │ customer_id ─────────┼──┘    │ volume_cm3   │
                     │ product_id  ─────────┼───────►│ (generated)  │
                     │ seller_id   ─────────┼───┐   └──────────────┘
                     │ date_id     ─────────┼───┐
                     │ price / freight_value│   │
                     │ payment_value/type   │   │      dim_date
                     │ payment_installments │   │   ┌──────────────┐
                     │ review_score         │   └──►│ date_id      │
                     │ order_status         │       │ day/month/q  │
                     │ delivered_customer_  │       │ year/dow/    │
                     │   date               │       │ is_weekend   │
                     │ estimated_delivery_  │       └──────────────┘
                     │   date               │
                     └──────────────────────┘

  grain: one row per order line (order_id, order_item_id)
  every order-level measure (payment, review, delivery) is denormalised
  onto the line so a single join-set powers revenue / satisfaction / SLA
```

### Why a star schema?

A star schema denormalises the normalised Olist source into **one fact table
surrounded by dimensions**, with each fact↔dim join on a single key. The
payoff:

- **Query speed.** Each analytical question (revenue by category, satisfaction
  by state, on-time delivery by month) is a fact table plus the one or two
  dimensions it needs — a single, shallow join-set, all FK-indexed. No long
  chains of normalised tables to walk.
- **BI-friendly.** Tools like Tableau / Metabase / Power BI read a star
  natively: the fact is the measure source, dimensions are the slice/dice
  axes. Users pick attributes without writing joins.
- **Consistent grain.** Denormalising payments/reviews onto each order line
  fixes the grain mismatch the EDA flagged (orders ≠ payment rows ≠ review
  rows). One fact grain = one unambiguous revenue number.
- **Read-optimised.** Storage is duplicated to buy read simplicity — the
  classic OLAP trade-off, and the right one for an analytics warehouse that
  is queried far more than it is written.

**Trade-offs acknowledged.** Denormalisation duplicates data (e.g. the
customer/state repeats on every line) so storage is larger and updates are
harder — acceptable here because the warehouse is append-mostly and query
latency is the priority.